In [1]:
import json
import re
from collections import defaultdict
from functools import cache
from typing import Annotated, Literal, Self

import numpy as np
import pandas as pd
import pulp
from pydantic import BaseModel, BeforeValidator, ConfigDict, TypeAdapter, computed_field

In [2]:
bracket_raw = pd.read_csv("00_bracket.csv")
lookup_raw = pd.read_csv("00_lookup.csv")
ratings_raw = pd.read_csv("01_ratings.csv")
tips_raw = pd.read_csv("02_tips.csv")
group_stage_points = pd.read_csv("../04_points.csv", index_col="name")["group_stage_points"].to_dict()

In [3]:
tips = tips_raw.groupby(["player", "stage"])["team"].apply(list).unstack().fillna("").apply(list)

In [4]:
ratings = (
    lookup_raw.merge(ratings_raw, how="left", left_on="eloratings_name", right_on="team")
    .set_index("fifa_code")["rating"]
    .to_dict()
)

In [5]:
def parse_team(s):
    if re.match(R"[WL]\d+", s):
        return PrevMatch.from_string(s)
    return s


class PrevMatch(BaseModel):
    match_number: int
    team: Literal["W", "L"]

    @classmethod
    def from_string(cls, s: str):
        return PrevMatch(match_number=int(s[1:]), team=s[0])


class BracketMatch(BaseModel):
    match_number: int
    home: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    away: Annotated[str | PrevMatch, BeforeValidator(parse_team)]
    stage: str
    winner: str | None = None

In [6]:
matches = {
    match.match_number: match
    for match in TypeAdapter(list[BracketMatch]).validate_python(
        bracket_raw.replace({np.nan: None}).to_dict(orient="records")
    )
}

In [7]:
class MatchTeam(BaseModel):
    team: str
    source: PrevMatch | None = None

    def replace_source(self, source: PrevMatch) -> Self:
        return self.model_copy(update={"source": source})


@cache
def possible_teams_for_match(match_number: int) -> list[MatchTeam]:
    match = matches[match_number]
    out = []
    if not isinstance(match.home, str):
        out.extend(mt.replace_source(match.home) for mt in possible_teams_for_match(match.home.match_number))
    else:
        out.append(MatchTeam(team=match.home))

    if not isinstance(match.away, str):
        out.extend(mt.replace_source(match.away) for mt in possible_teams_for_match(match.away.match_number))
    else:
        out.append(MatchTeam(team=match.away))

    return out


def var_name(match, team, kind):
    return f"m{match:03d}_{team}_{kind}"

In [8]:
base_prob = pulp.LpProblem("Bracket", pulp.LpMaximize)

winner_vars = defaultdict(dict)
participant_vars = defaultdict(dict)
participants_by_stage = {match.stage: defaultdict(list) for match in matches.values()}
winners_by_stage = {match.stage: defaultdict(list) for match in matches.values()}

STAGE_WEIGHT = {"third": 0.5}

ratings_difference = 0

for match, bracket_match in sorted(matches.items()):
    match_winners = []
    match_participants = []
    for match_team in possible_teams_for_match(match):
        team = match_team.team
        source = match_team.source
        rating = ratings[team]
        winner_var = base_prob.add_variable(var_name(match, team, "winner"), 0, 1, pulp.LpInteger)
        participant_var = base_prob.add_variable(var_name(match, team, "participant"), 0, 1, pulp.LpInteger)

        if bracket_match.winner == team:
            winner_var.setInitialValue(1)
            winner_var.fixValue()
            participant_var.setInitialValue(1)
            participant_var.fixValue()

        ratings_difference += STAGE_WEIGHT.get(bracket_match.stage, 1) * (2 * winner_var - participant_var) * rating

        base_prob += participant_var >= winner_var
        if source:
            source_part_var = participant_vars[source.match_number][team]
            base_prob += source_part_var >= participant_var
            source_winner_var = winner_vars[source.match_number][team]
            if source.team == "W":
                base_prob += source_winner_var == participant_var
            else:
                base_prob += (1 - source_winner_var) >= participant_var
        else:
            base_prob += participant_var == 1

        winner_vars[match][team] = winner_var
        participant_vars[match][team] = participant_var

        stage = bracket_match.stage
        winners_by_stage[stage][team].append(winner_var)
        participants_by_stage[stage][team].append(participant_var)

        match_winners.append(winner_var)
        match_participants.append(participant_var)
    base_prob += pulp.lpSum(match_winners) == 1
    base_prob += pulp.lpSum(match_participants) == 2

base_prob += pulp.LpVariable("ratings_difference", cat=pulp.LpContinuous) == ratings_difference

winner_vars = dict(winner_vars)
participant_vars = dict(participant_vars)

In [9]:
PARTICIPANT_SCORING = {"R32": 1, "R16": 2, "quarter": 4, "semi": 6, "final": 8}
WINNER_SCORING = {"third": ("third", 4), "final": ("winner", 12)}

In [10]:
players = sorted(tips_raw["player"].unique())

player_vars = base_prob.add_variable_dict("player_score", players, cat=pulp.LpInteger)
variables_dict = base_prob.variablesDict()
for player in players:
    player_score = group_stage_points[player]
    for match in matches.values():
        stage = match.stage
        if stage in PARTICIPANT_SCORING:
            participant_points = PARTICIPANT_SCORING[stage]
            participant_teams = tips.loc[player, stage]
            player_score += participant_points * pulp.lpSum(
                [variables_dict.get(var_name(match.match_number, team, "participant"), 0) for team in participant_teams]
            )
        if stage in WINNER_SCORING:
            winner_stage, winner_pts = WINNER_SCORING[stage]
            winner_teams = tips.loc[player, winner_stage]
            player_score += winner_pts * pulp.lpSum(
                [
                    base_prob.variablesDict().get(var_name(match.match_number, team, "winner"), 0)
                    for team in winner_teams
                ]
            )
    base_prob += player_vars[player] == player_score

player_name_to_var_name = {name: var.name for name, var in player_vars.items()}

In [11]:
class SolvedMatch(BaseModel):
    model_config = ConfigDict(extra="forbid")

    match_number: int
    stage: str
    home: str
    home_rating: int
    away: str
    away_rating: int
    home_winner: bool
    known_result: bool

    @computed_field
    def upset(self) -> bool:
        return (self.home_rating < self.away_rating) == self.home_winner

    def to_string(self):
        return f"{self.match_number:3} {self.stage:7} {'👑' if self.home_winner else '  '}{self.home} ({self.home_rating}) vs {'👑' if not self.home_winner else '  '}{self.away} ({self.away_rating})"


class PlayerResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str
    score: int


class Results(BaseModel):
    model_config = ConfigDict(extra="forbid")

    matches: list[SolvedMatch]
    players: list[PlayerResult]


class ResultSet(BaseModel):
    model_config = ConfigDict(extra="forbid")

    spec: list[str]
    results: Results | None

In [12]:
def get_results(solved_prob):
    winners = {}
    participants = defaultdict(list)

    for var in solved_prob.variables():
        if var.name == "__dummy":
            continue
        if var.value() != 1:
            continue
        match_raw, team, kind = var.name.split("_")
        match = int(match_raw[1:])
        if kind == "winner":
            winners[match] = team
        elif kind == "participant":
            participants[match].append(team)
        else:
            raise ValueError(var.name)

    participants = dict(participants)

    solved_matches = []

    for match in sorted(matches):
        winner = winners[match]
        home, away = participants[match]
        home_winner = winner == home
        solved_matches.append(
            SolvedMatch(
                match_number=match,
                stage=matches[match].stage,
                home=home,
                away=away,
                home_winner=home_winner,
                home_rating=ratings[home],
                away_rating=ratings[away],
                known_result=bool(matches[match].winner),
            )
        )

    variables_dict = solved_prob.variablesDict()
    player_results = [
        PlayerResult(name=name, score=variables_dict[var_name].value())
        for name, var_name in sorted(player_name_to_var_name.items())
    ]

    return Results(matches=solved_matches, players=player_results)

In [13]:
def get_ratings_difference(pulp_problem):
    return pulp_problem.variablesDict()["ratings_difference"]


def optimise_by_ratings_difference(pulp_problem):
    pulp_problem += get_ratings_difference(pulp_problem)

In [14]:
result_sets: list[ResultSet] = []

no_upsets = base_prob.copy()
optimise_by_ratings_difference(no_upsets)
no_upsets.solve(pulp.PULP_CBC_CMD(msg=False))

result_sets.append(ResultSet(spec=["No upsets"], results=get_results(no_upsets)))

for player_name, player_var_name in player_name_to_var_name.items():
    player_winner = base_prob.copy()
    player_winner_vars = player_winner.variablesDict()
    for opponent_var_name in player_name_to_var_name.values():
        if player_var_name == opponent_var_name:
            continue
        player_winner += player_winner_vars[player_var_name] >= player_winner_vars[opponent_var_name]

    optimise_by_ratings_difference(player_winner)
    solve_status = player_winner.solve(pulp.PULP_CBC_CMD(msg=False))

    if pulp.LpStatus[solve_status] == "Infeasible":
        result_sets.append(ResultSet(spec=[player_name, "Winner"], results=None))
    else:
        result_sets.append(ResultSet(spec=[player_name, "Winner"], results=get_results(player_winner)))

    player_max_score = base_prob.copy()
    player_max_score += 100 * player_winner.variablesDict()[player_var_name] + get_ratings_difference(player_max_score)
    player_max_score.solve(pulp.PULP_CBC_CMD(msg=False))
    result_sets.append(ResultSet(spec=[player_name, "Most points possible"], results=get_results(player_max_score)))

    player_loser = base_prob.copy()
    player_loser_vars = player_loser.variablesDict()
    for opponent_var_name in player_name_to_var_name.values():
        if player_var_name == opponent_var_name:
            continue
        player_loser += player_loser_vars[player_var_name] <= player_loser_vars[opponent_var_name]

    optimise_by_ratings_difference(player_loser)
    solve_status = player_loser.solve(pulp.PULP_CBC_CMD(msg=False))

    if pulp.LpStatus[solve_status] == "Infeasible":
        result_sets.append(ResultSet(spec=[player_name, "Loser"], results=None))
    else:
        result_sets.append(ResultSet(spec=[player_name, "Loser"], results=get_results(player_loser)))

    player_min_score = base_prob.copy()
    player_min_score += -100 * player_loser.variablesDict()[player_var_name] + get_ratings_difference(player_min_score)
    player_min_score.solve(pulp.PULP_CBC_CMD(msg=False))
    result_sets.append(ResultSet(spec=[player_name, "Least points possible"], results=get_results(player_min_score)))

result_set_adapter = TypeAdapter(list[ResultSet])

with open("03_results.json", "wb") as f:
    f.write(result_set_adapter.dump_json(result_sets, indent=2))

In [15]:
with open("03_results_schema.json", "w") as f:
    json.dump(ResultSet.model_json_schema(), f, indent=2)

In [16]:
! cp 03_results.json ../next-app/src/data/results.json